# EPL Result Modeling: Elo Baseline and Stacked XGBoost

This notebook uses `epl_model_features.csv` to:

1. fit an Elo-only multinomial logistic regression baseline
2. generate held-out Elo probabilities for `2024/25`
3. train a stacked XGBoost model using `EloProb_*` plus pre-match recent-form features
4. compare multiclass log loss and accuracy

The split is time-based and not shuffled:

- Train: `2020/21` through `2023/24`
- Validation/Test: `2024/25`

For the stacked model, the validation Elo probabilities come from a logistic regression trained only on the training split. The XGBoost training rows use expanding-window out-of-fold Elo probabilities by season so the stack stays time-aware.

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RESULT_ORDER = ["H", "D", "A"]
RESULT_TO_INT = {result: idx for idx, result in enumerate(RESULT_ORDER)}
INT_TO_RESULT = {idx: result for result, idx in RESULT_TO_INT.items()}

TRAIN_SEASONS = ["2020/21", "2021/22", "2022/23", "2023/24"]
VALID_SEASON = "2024/25"

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent

DATA_PATH = project_root / "data" / "epl_model_features.csv"
DATA_PATH

PosixPath('/home/jack$/soccer/data/epl_model_features.csv')

In [21]:
df = pd.read_csv(DATA_PATH, na_values=["NaN"]).copy()
df["MatchDate"] = pd.to_datetime(df["MatchDate"])

numeric_columns = [
    "EloDiff",
    "Home_GoalsScored_Last5",
    "Away_GoalsScored_Last5",
    "GoalsConceded_Last5_Diff",
    "Shots_Last5_Diff",
    "SOT_Last5_Diff",
    "Home_ShotsAllowed_Last5",
    "Away_ShotsAllowed_Last5",
    "Home_SOTAllowed_Last5",
    "Away_SOTAllowed_Last5",
    "PPG_Last5_Diff",
    "RestDiff",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["GoalsScored_Last5_Diff"] = (
    df["Home_GoalsScored_Last5"] - df["Away_GoalsScored_Last5"]
)
df["ShotsAllowed_Last5_Diff"] = (
    df["Home_ShotsAllowed_Last5"] - df["Away_ShotsAllowed_Last5"]
)
df["SOTAllowed_Last5_Diff"] = (
    df["Home_SOTAllowed_Last5"] - df["Away_SOTAllowed_Last5"]
)
df["target"] = df["FullTimeResult"].map(RESULT_TO_INT)

display(df[["Season", "MatchDate", "HomeTeam", "AwayTeam", "EloDiff", "PPG_Last5_Diff", "GoalsScored_Last5_Diff", "ShotsAllowed_Last5_Diff", "RestDiff", "FullTimeResult"]].head(10))
display(df["Season"].value_counts(sort=False).rename("Matches"))

,Season,MatchDate,HomeTeam,AwayTeam,EloDiff,PPG_Last5_Diff,GoalsScored_Last5_Diff,ShotsAllowed_Last5_Diff,RestDiff,FullTimeResult
0,2020/21,2020-09-12,Fulham,Arsenal,0.0,NaN,NaN,NaN,NaN,A
1,2020/21,2020-09-12,Crystal Palace,Southampton,0.0,NaN,NaN,NaN,NaN,H
2,2020/21,2020-09-12,Liverpool,Leeds,0.0,NaN,NaN,NaN,NaN,H
3,2020/21,2020-09-12,West Ham,Newcastle,0.0,NaN,NaN,NaN,NaN,A
4,2020/21,2020-09-13,West Brom,Leicester,0.0,NaN,NaN,NaN,NaN,A
5,2020/21,2020-09-13,Tottenham,Everton,0.0,NaN,NaN,NaN,NaN,A
6,2020/21,2020-09-14,Brighton,Chelsea,0.0,NaN,NaN,NaN,NaN,A
7,2020/21,2020-09-14,Sheffield United,Wolves,0.0,NaN,NaN,NaN,NaN,A
8,2020/21,2020-09-19,Everton,West Brom,20.0,3.0,1.0,-4.0,0.0,H
9,2020/21,2020-09-19,Leeds,Fulham,0.0,0.0,3.0,9.0,0.0,H


Season
2020/21    380
2021/22    380
2022/23    380
2023/24    380
2024/25    350
Name: Matches, dtype: int64

## Step 1: Elo Baseline Regression

Baseline model:

- input: `EloDiff`
- target: `FullTimeResult` with classes `H`, `D`, `A`
- model: multinomial logistic regression

This produces smooth baseline probabilities:

- `EloProb_H`
- `EloProb_D`
- `EloProb_A`

In [22]:
train_df = df.loc[df["Season"].isin(TRAIN_SEASONS)].copy()
valid_df = df.loc[df["Season"] == VALID_SEASON].copy()

baseline_features = ["EloDiff"]

baseline_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)
baseline_model.fit(train_df[baseline_features], train_df["target"])


valid_baseline_probs = baseline_model.predict_proba(valid_df[baseline_features])
valid_df[["EloProb_H", "EloProb_D", "EloProb_A"]] = valid_baseline_probs
valid_df["EloPred"] = valid_baseline_probs.argmax(axis=1)
valid_df["EloPredLabel"] = valid_df["EloPred"].map(INT_TO_RESULT)

baseline_metrics = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression (Elo only)",
            "LogLoss": log_loss(valid_df["target"], valid_baseline_probs, labels=[0, 1, 2]),
            "Accuracy": accuracy_score(valid_df["target"], valid_df["EloPred"]),
        }
    ]
)

display(baseline_metrics)
display(valid_df[["Season", "MatchDate", "HomeTeam", "AwayTeam", "EloDiff", "EloProb_H", "EloProb_D", "EloProb_A", "FullTimeResult", "EloPredLabel"]].head(20))

,Model,LogLoss,Accuracy
0,Logistic Regression (Elo only),0.992085,0.52


,Season,MatchDate,HomeTeam,AwayTeam,EloDiff,EloProb_H,EloProb_D,EloProb_A,FullTimeResult,EloPredLabel
1520,2024/25,2024-08-16,Man United,Fulham,101.29,0.615908,0.205682,0.178410,H,H
1521,2024/25,2024-08-17,Ipswich,Liverpool,-195.66,0.142651,0.215737,0.641611,A,A
1522,2024/25,2024-08-17,Arsenal,Wolves,261.84,0.824177,0.121630,0.054193,H,H
1523,2024/25,2024-08-17,Everton,Brighton,-20.24,0.398469,0.246909,0.354622,A,H
1524,2024/25,2024-08-17,Newcastle,Southampton,211.10,0.771549,0.147391,0.081060,H,H
1525,2024/25,2024-08-17,Nott'm Forest,Bournemouth,-44.20,0.355828,0.249064,0.395108,D,A
1526,2024/25,2024-08-17,West Ham,Aston Villa,-90.38,0.278839,0.246852,0.474309,A,A
1527,2024/25,2024-08-18,Brentford,Crystal Palace,-39.61,0.363887,0.248827,0.387286,H,A
1528,2024/25,2024-08-18,Chelsea,Man City,-196.71,0.141603,0.215298,0.643100,A,A
1529,2024/25,2024-08-19,Leicester,Tottenham,-127.47,0.223961,0.239436,0.536603,D,A


## Step 2: XGBoost With Extra Features

Stacked setup:

- baseline Elo model provides `EloProb_H`, `EloProb_D`, `EloProb_A`
- XGBoost consumes those probabilities plus recent-game pre-match features

To keep the stack honest, the training rows get season-based expanding-window out-of-fold Elo probabilities. That means the first training season (`2020/21`) has no prior-season baseline model available and is excluded from the stacked XGBoost training fit.

In [ ]:
def make_probability_frame(probabilities, index, prefix):
    columns = [f"{prefix}_H", f"{prefix}_D", f"{prefix}_A"]
    return pd.DataFrame(probabilities, index=index, columns=columns)


def expanding_window_elo_oof(train_frame, seasons, feature_name="EloDiff"):
    oof_frames = []

    for season_idx in range(1, len(seasons)):
        fit_seasons = seasons[:season_idx]
        score_season = seasons[season_idx]

        fit_frame = train_frame.loc[train_frame["Season"].isin(fit_seasons)]
        score_frame = train_frame.loc[train_frame["Season"] == score_season].copy()

        model = LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            random_state=42,
        )
        model.fit(fit_frame[[feature_name]], fit_frame["target"])
        score_probs = model.predict_proba(score_frame[[feature_name]])
        score_frame[["EloProb_H", "EloProb_D", "EloProb_A"]] = score_probs
        oof_frames.append(score_frame)

    stacked_train = pd.concat(oof_frames).sort_index()
    return stacked_train


stack_train_df = expanding_window_elo_oof(train_df, TRAIN_SEASONS)
stack_valid_df = valid_df.copy()

stack_feature_columns = [
    "EloProb_H",
    "EloProb_D",
    "EloProb_A",
    "PPG_Last5_Diff",
    "GoalsScored_Last5_Diff",
    "GoalsConceded_Last5_Diff",
    "Shots_Last5_Diff",
    "SOT_Last5_Diff",
    "ShotsAllowed_Last5_Diff",
    "SOTAllowed_Last5_Diff",
    "RestDiff",
]

xgb_train_missing_mask = stack_train_df[stack_feature_columns + ["target"]].isna().any(axis=1)
xgb_train_df = stack_train_df.loc[~xgb_train_missing_mask].copy()

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=300,
    learning_rate=0.03,
    max_depth=2,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
)

xgb_model.fit(xgb_train_df[stack_feature_columns], xgb_train_df["target"])

valid_xgb_probs = xgb_model.predict_proba(stack_valid_df[stack_feature_columns])
stack_valid_df[["XGBProb_H", "XGBProb_D", "XGBProb_A"]] = valid_xgb_probs
stack_valid_df["XGBPred"] = valid_xgb_probs.argmax(axis=1)
stack_valid_df["XGBPredLabel"] = stack_valid_df["XGBPred"].map(INT_TO_RESULT)

xgb_metrics = pd.DataFrame(
    [
        {
            "Model": "XGBoost (stacked on Elo probabilities)",
            "LogLoss": log_loss(stack_valid_df["target"], valid_xgb_probs, labels=[0, 1, 2]),
            "Accuracy": accuracy_score(stack_valid_df["target"], stack_valid_df["XGBPred"]),
        }
    ]
)

print(f"Stacked XGBoost training rows: {len(xgb_train_df):,}")
print(f"Rows dropped from stacked training due to no prior-season Elo baseline: {len(train_df) - len(stack_train_df):,}")
print(f"Rows dropped from stacked training due to missing feature data: {xgb_train_missing_mask.sum():,}")

Stacked XGBoost training rows: 1,134
Rows dropped from stacked training due to no prior-season Elo baseline: 380
Rows dropped from stacked training due to missing feature data: 6


In [30]:
comparison = pd.concat([baseline_metrics, xgb_metrics], ignore_index=True)
display(comparison)

validation_scored = stack_valid_df[
    [
        "Season",
        "MatchDate",
        "HomeTeam",
        "AwayTeam",
        "FullTimeResult",
        "EloDiff",
        "EloProb_H",
        "EloProb_D",
        "EloProb_A",
        "XGBProb_H",
        "XGBProb_D",
        "XGBProb_A",
        "EloPredLabel",
        "XGBPredLabel",
    ]
].copy()

display(validation_scored.head(10))





,Model,LogLoss,Accuracy
0,Logistic Regression (Elo only),0.992085,0.520000
1,XGBoost (stacked on Elo probabilities),1.020038,0.522857


,Season,MatchDate,HomeTeam,AwayTeam,FullTimeResult,EloDiff,EloProb_H,EloProb_D,EloProb_A,XGBProb_H,XGBProb_D,XGBProb_A,EloPredLabel,XGBPredLabel
1520,2024/25,2024-08-16,Man United,Fulham,H,101.29,0.615908,0.205682,0.178410,0.577270,0.248423,0.174307,H,H
1521,2024/25,2024-08-17,Ipswich,Liverpool,A,-195.66,0.142651,0.215737,0.641611,0.382432,0.083836,0.533733,A,A
1522,2024/25,2024-08-17,Arsenal,Wolves,H,261.84,0.824177,0.121630,0.054193,0.931162,0.035185,0.033653,H,H
1523,2024/25,2024-08-17,Everton,Brighton,A,-20.24,0.398469,0.246909,0.354622,0.238009,0.197432,0.564559,H,A
1524,2024/25,2024-08-17,Newcastle,Southampton,H,211.10,0.771549,0.147391,0.081060,0.821995,0.127892,0.050113,H,H
1525,2024/25,2024-08-17,Nott'm Forest,Bournemouth,D,-44.20,0.355828,0.249064,0.395108,0.521456,0.171761,0.306783,A,H
1526,2024/25,2024-08-17,West Ham,Aston Villa,A,-90.38,0.278839,0.246852,0.474309,0.275814,0.247044,0.477142,A,A
1527,2024/25,2024-08-18,Brentford,Crystal Palace,H,-39.61,0.363887,0.248827,0.387286,0.254122,0.255836,0.490042,A,A
1528,2024/25,2024-08-18,Chelsea,Man City,A,-196.71,0.141603,0.215298,0.643100,0.195479,0.155063,0.649458,A,A
1529,2024/25,2024-08-19,Leicester,Tottenham,D,-127.47,0.223961,0.239436,0.536603,0.093783,0.152438,0.753780,A,A
